<center>
    <h1></h1>
    <h1><b>OCES 5303</b></h1>
    <h2>Assignment 1</h2>
    <hr>
    <p>Jonas Mathisrud Sterud</p>
    <p>21335836</p>
</center>

<center>
    <h5><i>Abstract</i></h5>
    <p>
    Argo is an international program collecting data from the global ocean at various depths using robotic instruments drifting with ocean currents.
    <small style="margin-left: 1em">[1]</small>
    </p>
    <p>
    In this assignment, we will explore the dataset, and perform linear regression  to see if we can predict the sea temperature.
    </p>
    <img src="./figures/cover.jpg" width="50%">
</center>

<h1>The Data</h1>

<p>
The standard mission for Argo's robotic instruments is what's known as "park-and-profile".
</p>

<p>
It involves the robotic instrument, the "float", descending to a depth of approximatly 1 km (1000 dbar) and
drifting with the ocean currents. Then, every 10 days, the float descents to 2 km (2000 dbar) and starts its data collection.
It collects data on its accent back up to the surface at various depths, taking around six hours to complete.
At the surface, the float transmits the data collected over satelitte.
</p>

<p>
Most of the floats record the temperature and the salinity of the water, and some also collect information on properties describing the biology/chemistry of the water.
In this report, we'll only look at the temperature and the salinity at various depths.
</p>

<small style="margin-left: 1em">[2] [3]</small>

<center>
    <img src="./figures/mission.jpg" width="50%">
</center>

<small style="margin-left: 1em">[7]</small>

In [ ]:
#####################################
#       You might need to restart   #
#           the kernel after        #
#           running this cell.      #
#####################################

## Detect environment

try:
    import google.colab # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

## Clone repository and download dependencies
## (assumes using Conda if not in Google Colab)

if IN_COLAB:
    None
    !git clone https://github.com/jonassterud/OCES5303_A1.git
    !cp -r OCES5303_A1/* .
    %pip install -r ./requirements.txt
else:
    None
    %conda install -c conda-forge --file ./requirements.txt

In [ ]:
## Imports

import xarray as xr
import numpy as np
import pandas as pd
import plotly.io as pio
import plotly.express as px
import warnings
import gc

from plotly.subplots import make_subplots
from sklearn import set_config
from sklearn.preprocessing import MinMaxScaler, StandardScaler, FunctionTransformer, OneHotEncoder, PolynomialFeatures
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, KFold
from sklearn.dummy import DummyRegressor
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score, PredictionErrorDisplay
from sklearn.cluster import KMeans
from scipy.linalg import LinAlgWarning

## Configuration

set_config(transform_output = "pandas")
pio.templates.default = "plotly_dark"
seed = 1010011_01110000_01100001_01101110_01101001_01110011_01101000_00100000_01001001_01101110_01110001_01110101_01101001_01110011_01101001_01110100_01101001_01101111_01101110_00100001 % 42

<h1>Parsing</h1>

<p>
We have six different variables, described in their attributes as:
</p>

<ul>
    <li><code>DEPTH</code>: Vertical Distance Below the Surface (m)</li>
    <li><code>PSAL</code>: Practical Salinity (PSU)</li>
    <li><code>TEMP</code>: Sea Temperature In-Situ ITS-90 Scale (℃)</li>
    <li><code>SIG0</code>: Potential Density Referenced to Surface (kg / m³)</li>
    <li><code>BRV2</code>: Brunt-Vaisala Frequency Squared (1/s²)<b>*</b></li>
    <li><code>TIME</code>: Time of Measurement</li>
    <li><code>LATITUDE</code>: Latitude (degrees north)</li>
    <li><code>LONGITUDE</code>: Longitude (degrees east)</li>
</ul>

<blockquote>
<b>*</b> "(...) measure of the stability of a fluid to vertical displacements such as those caused by convection."
</blockquote>

<small style="margin-left: 1em">[6]</small>

<p>
To make our data be more compatible with <i>Plotly Express</i> and <i>scikit-learn</i>, we'll also use <i>Pandas</i>' <code>DataFrame</code>.
</p>

<p>
Here, we'll only look at data from the Atlantic Ocean, so to start off, we'll remove rows with latitude and longitude outside <code>(-75, 17)</code>. In addition, we remove rows where the salinity range falls outside reasonable values.
</p>

In [ ]:
## Read and process data

ds = xr.open_zarr("data/GLOB_HOMOGENEOUS_variables.zarr/", consolidated=False)
da_al = ds[["PSAL", "TEMP", "DEPTH", "SIG0", "BRV2"]] # (DEPTH: 302, N_PROF: 128910)
da_s  = da_al.where(((da_al.PSAL < 40.) & (da_al.PSAL > 25.)).compute(), drop=True) # Remove rows outside reasonable salinity range
da_s  = da_s.dropna('N_PROF')
da = da_s.where((da_s['LONGITUDE'] > -75.) & (da_s['LONGITUDE'] < 17), drop= True) # Atlantic ocean

df = da.to_dataframe().reset_index()
df["TIME"] = df["TIME"].to_numpy(float)
df["LATITUDE"] = df["LATITUDE"].to_numpy(float)
df["LONGITUDE"] = df["LONGITUDE"].to_numpy(float)

<h2>Split Data</h2>

<p>
Let's also split the data up into three different data sets: training, validation and testing. We do this to avoid any biases in our training, and minimize overfitting. Let's also drop the <code>N_PROF</code> variable, since this is essentialy an identification variable for each profile - and won't generalize for new Argo data.
</p>

In [ ]:
## Split data (70%, 15%, 15%)

X = df.drop(columns = ["N_PROF", "TEMP"])
y = df["TEMP"]

X_train, X_val_test, y_train, y_val_test = train_test_split(X, y, train_size = 0.7, random_state = seed)
X_val, X_test, y_val, y_test = train_test_split(X_val_test, y_val_test, train_size = 0.5, test_size = 0.5, random_state = seed)

## Combined train set (for visualization)

Xy_train = pd.concat([X_train, y_train], axis=1)

## Create a subset of data used for visualization, with scaled features

df_viz = Xy_train.sample(300, random_state=seed)
for feat in df_viz.columns:
    scaler = StandardScaler().fit(Xy_train[feat].values.reshape(-1, 1))
    df_viz[f"{feat}_S"] = scaler.transform(df_viz[feat].values.reshape(-1, 1)).values

# Manually clean up

%xdel X
%xdel y
%xdel X_val_test
%xdel y_val_test
%xdel Xy_train
%xdel df

<h1>Exploratory Data Analysis</h1>

<p>Now, let's explore the data to see if we can find any outliers, trends, etc. Let's start by plotting some of the geographical points in our dataset, colored with the values of <code>TEMP</code>, <code>PSAL</code>, <code>SIG0</code> and <code>BRV2</code>.</p>

In [ ]:
## Create figures

fig_geo_temp = px.scatter_geo(df_viz, lat="LATITUDE", lon="LONGITUDE", color="TEMP_S", hover_name="TEMP", hover_data=["DEPTH"])
fig_geo_psal = px.scatter_geo(df_viz, lat="LATITUDE", lon="LONGITUDE", color="PSAL_S", hover_name="PSAL", hover_data=["DEPTH"])
fig_geo_sig0 = px.scatter_geo(df_viz, lat="LATITUDE", lon="LONGITUDE", color="SIG0_S", hover_name="SIG0", hover_data=["DEPTH"])
fig_geo_brv2 = px.scatter_geo(df_viz, lat="LATITUDE", lon="LONGITUDE", color="BRV2_S", hover_name="BRV2", hover_data=["DEPTH"])

fig_geo = make_subplots(rows=2, cols=2, specs=[[{'type': 'geo'}, {'type': 'geo'}], [{'type': 'geo'}, {'type': 'geo'}]], subplot_titles=("Temperature", "Salinity", "Density", "Stability"))
for trace in fig_geo_temp.data: fig_geo.add_trace(trace, row=1, col=1)
for trace in fig_geo_psal.data: fig_geo.add_trace(trace, row=1, col=2)
for trace in fig_geo_sig0.data: fig_geo.add_trace(trace, row=2, col=1)
for trace in fig_geo_brv2.data: fig_geo.add_trace(trace, row=2, col=2)

fig_geo.update_layout(height=700, title_text="Locations of floats and their data", coloraxis_colorbar=dict(orientation="h", y=-0.2, yanchor="bottom", x=0.5, xanchor="center"), coloraxis=dict(colorscale="Plasma"))
fig_geo.show()

<p>
We see that our attempt at only selecting data from the Atlantic Ocean has failed, and we have some outliers in the Pacific Ocean.
</p>

<p>
These might interfere with our results, due to natural phenomenons such as ENSO (causing sea surface temperature differences in the Pacific Ocean). Let's remove any observations made with a longitude smaller than <code>-60.0</code> if the latitude is also smaller than <code>7.0</code>. In reality, these geographical restrictions aren't a perfect description of the Pacific Ocean, but for our data, it's good enough.
</p>

<small style="margin-left: 1em">[5]</small>

In [ ]:
## Remove geographical outliers

mask = X_train.index[(X_train["LONGITUDE"] < -60) & (X_train["LATITUDE"] < 7)]

X_train = X_train.drop(mask, axis=0)
y_train = y_train.drop(mask, axis=0)

<p>Other than that, we see a few points of what might look like outliers when compared to their neighbouring points, but we won't go into that here.</p>

<h1>Trends</h1>

<p>
Now, let's see if we can see any obvious trends between our features and our target variables.
We'll scale the data using <code>StandardScaler</code> and then plot the features against our target variable to see if we can spot any obvious patterns.
</p>

In [ ]:
feature_names = X_train.columns

# Create plots

fig_trends = make_subplots(rows=len(feature_names), cols=1, subplot_titles=list(map(lambda v: f"{v} vs. TEMP", feature_names)))

for (i, feat) in enumerate(feature_names):
    fig_trend = px.scatter(df_viz, x=f"{feat}_S", y=f"TEMP_S", color="TEMP_S", trendline="ols", hover_data=[feat, "TEMP"])
    fig_trend.update_traces(line={"color": "orange", "width": 2, "dash": "dot"})
    for trace in fig_trend.data: fig_trends.add_trace(trace, row=(i + 1), col=1)

fig_trends.update_layout(height=1500, coloraxis_showscale=False, coloraxis=dict(colorscale="Plasma"))
fig_trends.show()

<p>
We see that <code>PSAL</code> and <code>SIG0</code> are the features with the strongest correlation to <code>TEMP</code>. In addition, <code>DEPTH</code> seems to be correlated to <code>TEMP</code> for lower values (which of course makes sense). The <code>LATITUDE</code> also seems to be a good predictor for lower temperatures, when the latitude is less than approx. <code>-50</code> or greater than approx. <code>50</code> (i.e. close to the polar circles).
</p>

<h1>Clustering</h1>

<p>
From the visualizations above, we see that some of the features seem to be correlated with our target variable in a non-linear fashion.
</p>

<p>
Next, we'll try clustering some of the features individually, to see if we get any clusters that'll help us describe these non-linear correlations. We'll see if there's any interactions between variables that might help predict our target variable.
</p>

In [ ]:
feature_names = ["LATITUDE", "LONGITUDE", "DEPTH", "PSAL", "SIG0", "BRV2"]
cluster_amount = [8, 4, 3, 3, 4, 3]

## Create figure

fig_clusters = make_subplots(rows=len(feature_names), cols=1, subplot_titles=list(map(lambda v: f"{v} cluster", feature_names)))

for (i, feat) in enumerate(feature_names):
    ## Find clusters

    c_kmeans = KMeans(n_clusters=cluster_amount[i], random_state=seed)
    df_viz[f"{feat}_CLUSTER"] = c_kmeans.fit_predict(df_viz[[f"{feat}_S"]])

    ## Add to figure

    fig_cluster = px.scatter(df_viz, x=f"{feat}_S", y="TEMP_S", color=f"{feat}_CLUSTER", trendline="ols", hover_data=[feat, "TEMP"])
    fig_cluster.update_traces(line={"color": "orange", "width": 2, "dash": "dot"})
    for trace in fig_cluster.data: fig_clusters.add_trace(trace, row=(i + 1), col=1)

## Show scatter plots with cluster zones

fig_clusters.update_layout(height=1500, coloraxis_showscale=False, coloraxis=dict(colorscale="Plasma"))
fig_clusters.show()

<p>
We see that for <code>LATITUDE</code>, the first and last cluster have less variation, and they seem to be good predictors for lower temperatures (or at least that the temperatures are below the trendline for ordinary least squares regression). This is of course something that could be picked up by our models, but we'll have to test.
</p>

<p>
In addition, the leftmost cluster for <code>SIG0</code> has observations with a much larger variance from the trendline compared to the other clusters - which also might be useful information for our model.
</p>

<p>
But first, let's see if we can find any clusters between <code>LATITUDE</code> and <code>LONGITUDE</code> to see if that's perhaps a good predictor for <code>TEMP</code>. Let's also try between <code>DEPTH</code> and <code>PSAL</code>. Here it's especially important to standardize the values before attempting to find clusters, since <code>DEPTH</code> and <code>PSAL</code> are using different scales.
</p>


In [ ]:
feature_names = [["LATITUDE", "LONGITUDE"], ["DEPTH", "PSAL"], ["SIG0", "PSAL"]]
cluster_amount = [6, 6, 6]

for (i, [feat_1, feat_2]) in enumerate(feature_names):
    ## Find clusters

    c_kmeans = KMeans(n_clusters=cluster_amount[i], random_state=seed)
    df_viz[f"{feat_1}_{feat_2}_CLUSTER"] = c_kmeans.fit_predict(df_viz[[f"{feat_1}_S", f"{feat_2}_S"]])

    ## Show 3D scatter plots with cluster zones

    fig_cluster = px.scatter_3d(df_viz, x=f"{feat_1}_S", y=f"{feat_2}_S", z="TEMP_S", color=f"{feat_1}_{feat_2}_CLUSTER", hover_data=[feat_1, feat_2, "TEMP"])
    fig_cluster.update_layout(coloraxis_showscale=False, coloraxis=dict(colorscale="Plasma"))
    fig_cluster.show()

<p>
The first plot shows clusters which clearly have a lower average temperature, and adding these to our model might improve the performance.
</p>

<p>
The second plot also shows a clear relationship between its variables and its effect on the temperature.
</p>

<p>
The last plot doesn't tell us really anything new, we already know that both of the variables here have a strong correlation to the temperature.
</p>

<p>
Let's create some transformers which we'll use later that'll add these features.
</p>

In [ ]:
# Pick relevant features

feature_names = ["LATITUDE", "LONGITUDE", "DEPTH", "PSAL"]

# Scaler

scaler = StandardScaler()
scaler = scaler.fit(X_train[feature_names])

# K-Means LATITUDE

c_kmeans_latitude = KMeans(n_clusters=8, random_state=seed)
c_kmeans_latitude = c_kmeans_latitude.fit(scaler.transform(X_train[feature_names])[["LATITUDE"]])

# K-Means LATITUDE/LONGITUDE

c_kmeans_latitude_longitude = KMeans(n_clusters=6, random_state=seed)
c_kmeans_latitude_longitude = c_kmeans_latitude_longitude.fit(scaler.transform(X_train[feature_names])[["LATITUDE", "LONGITUDE"]])

# K-Means DEPTH/PSAL

c_kmeans_depth_psal = KMeans(n_clusters=6, random_state=seed)
c_kmeans_depth_psal = c_kmeans_depth_psal.fit(scaler.transform(X_train[feature_names])[["DEPTH", "PSAL"]])

## Cluster: LATITUDE (CL)

def _add_clusters_latitude(data):
    data_c = data.copy()
    X_c = scaler.transform(data[feature_names])[["LATITUDE"]]
    data_c["CL"] = c_kmeans_latitude.predict(X_c)

    return data_c

def _remove_latitude(data):
    data_c = data.copy()
    data_c = data_c.drop(columns=["LATITUDE"])

    return data_c

## Cluster: LATITUDE LONGITUDE (CLL) 

def _add_clusters_latitude_longitude(data):
    data_c = data.copy()
    X_c = scaler.transform(data[feature_names])[["LATITUDE", "LONGITUDE"]]
    data_c["CLL"] = c_kmeans_latitude_longitude.predict(X_c)

    return data_c

def _remove_longitude(data):
    data_c = data.copy()
    data_c = data_c.drop(columns=["LONGITUDE"])

    return data_c

## Cluster: DEPTH PSAL (CDP)

def _add_clusters_depth_psal(data):
    data_c = data.copy()
    X_c = scaler.transform(data[feature_names])[["DEPTH", "PSAL"]]
    data_c["CDP"] = c_kmeans_depth_psal.predict(X_c)

    return data_c

def _remove_depth_psal(data):
    data_c = data.copy()
    data_c = data_c.drop(columns=["DEPTH", "PSAL"])

    return data_c


## Create (intermediary) cluster transformers

_t_clusters_latitude = FunctionTransformer(func=_add_clusters_latitude)
_t_clusters_latitude_longitude = FunctionTransformer(func=_add_clusters_latitude_longitude)
_t_clusters_depth_psal = FunctionTransformer(func=_add_clusters_depth_psal)

_t_remove_latitude = FunctionTransformer(func=_remove_latitude)
_t_remove_longitude = FunctionTransformer(func=_remove_longitude)
_t_remove_depth_psal = FunctionTransformer(func=_remove_depth_psal)

## Create (intermediary) encoder transformers

_t_clusters_encode_latitude = ColumnTransformer([("clusters_encode_latitude", OneHotEncoder(sparse_output=False), ["CL"])], remainder="passthrough", verbose_feature_names_out=False)
_t_clusters_encode_latitude_longitude = ColumnTransformer([("clusters_encode_latitude_longitude", OneHotEncoder(sparse_output=False), ["CLL"])], remainder="passthrough", verbose_feature_names_out=False)
_t_clusters_encode_depth_psal = ColumnTransformer([("clusters_encode_depth_psal", OneHotEncoder(sparse_output=False), ["CDP"])], remainder="passthrough", verbose_feature_names_out=False)

## Create transformers for finding clusters (after scaling), encoding, and optionally removing features

t_clusters = Pipeline([
    ("1", _t_clusters_latitude),
    ("2", _t_clusters_latitude_longitude),
    ("3", _t_clusters_depth_psal),
    ("4", _t_clusters_encode_latitude),
    ("5", _t_clusters_encode_latitude_longitude),
    ("6", _t_clusters_encode_depth_psal)
])

t_clusters_remove_features = Pipeline([
    ("1", _t_remove_latitude),
    ("2", _t_remove_longitude),
    ("3", _t_remove_depth_psal),
])

<h1>Metrics</h1>

<p>
Let's define a function that prints various metrics for each model we train.
</p>

<p>
To avoid overfitting, or in other words, choosing a model that's too complex and heavily biased towards our data (but will struggle to generalize), we'll use cross validation. This will split the data into five different subsets, and get the scores for each subset. We'll use the average of these scores in our metrics.
</p>

In [ ]:
def metrics(model, visualize=True):
    out = dict({
        "Mean Absolute Error": np.mean(cross_val_score(model, X_val, y_val, scoring="neg_mean_absolute_error")),
        "Root Mean Squared Error": np.mean(cross_val_score(model, X_val, y_val, scoring="neg_root_mean_squared_error")),
        "R2": np.mean(cross_val_score(model, X_val, y_val, scoring="r2"))
    })

    for k, v in out.items():
        print(f"{k}: {np.abs(v):.5f}")

    if (visualize): return PredictionErrorDisplay.from_estimator(model, X_val, y_val)

We'll mainly use the R² metric to check the performance of our model. The R² tells us the proportion between the residuals and the variance in our data. It can be a useful metric in that it also includes the variance in our data as a factor.

<center>
    <img src="./figures/r2.jpg" width="30%">
</center>

<small style="margin-left: 1em">[9]</small>

<h1>Baseline Model</h1>

<p>
Now that we have found possible features to include and created a way to analyze the performance of our models, we
re ready to begin trying different regression models.
</p>

<p>
But first, we'll train a baseline model so we have something to compare our models against.
The baseline model will simply calculate the mean of our target variable.
</p>

In [ ]:
## Train baseline model

r_dummy = DummyRegressor()
r_dummy = r_dummy.fit(X_train, y_train)

## Visualize

_ = metrics(r_dummy, False)

<h1>Linear Regression</h1>

<p>
Let's see if linear regression performs any better.
</p>

In [ ]:
## Train linear regression model

r_linear = LinearRegression()
r_linear = r_linear.fit(X_train, y_train)

## Visualize

_ = metrics(r_linear, False)

<p>
Perhaps a bit suprisingly, the linear regression struggles to find any relationship between the features and the target variable, and our performance results are almost identical to our <code>DummyRegressor</code>.
</p>

<p>
Let's try a few other models.
</p>

<h1>Random Forest Regression</h1>

<p>
Random Forest regression is a model which uses multipile decision tree regressors on subsets of the data, and averages the results (similar to cross validation). It's a scale invariant model, meaning that it won't impact performance if our variables are in differnet scales (which they are).
</p>

<p>
Running this in Google Colab is sometimes very slow, so we skip this model if it's running in that environment.
</p>

In [ ]:
## Train random forest regression model

if IN_COLAB == False:
    r_randomforest = RandomForestRegressor(n_estimators=10, max_depth=5, n_jobs=-1, random_state=seed)
    r_randomforest = r_randomforest.fit(X_train, y_train)

    ## Visualize

    _ = metrics(r_randomforest, False)

<h1>Ridge Regression</h1>

<p>
Now, let's test Ridge regression.
</p>

<p>
This regression model, like <code>LinearRegression</code>, also uses least squares, but includes L2 regularization, which is a penalty term rewarding less complex models.
</p>

<p>
We get the results:
</p>

<ul>
    <li>Mean Absolute Error: 0.64651</li>
    <li>Root Mean Squared Error: 0.90386</li>
    <li>R2: 0.97237</li>
</ul>

In [ ]:
warnings.filterwarnings("ignore", category=LinAlgWarning)

## Train Ridge regression model

r_ridge = Ridge(random_state=seed)
r_ridge = r_ridge.fit(X_train, y_train)

## Visualize

_ = metrics(r_ridge, False)

<p>
The results for Random Forest and Ridge regression are much better!
</p>

<p>
Ridge regression and Random Forest regression both perform almost the same.
</p>

<p>
Due to the time it takes to run the Random Forest model, and the similar performance, we'll continue using Ridge regression and try to see if we can make it perform even better.
</p>

<h1>Pipeline, Hyperparameters & Overfitting</h1>

<p>
Now we'll create a pipeline where we can test various preprocessing steps and check whether they improve/worsen our predictions.
</p>

<p>
In addition, we'll use <code>GridSearchCV</code> to test various preprocessing steps and hyperparameters on subsets (folds) of data. Here, we'll also use the validation set, to minimize overfitting on the training set.
</p>

<p>
For the hyperparameter search, using <code>GridSearchCV</code>, we'll use 5-fold cross-validation, with shuffling. That involves splitting the data into five different shuffled (random) subsets, where four of them are used for training and the last fold is used for validation. It repeates this process five times, so each fold gets one iteration being the validation set.
</p>

<p>
The performance of a hyperparameter configuration is then calculated as the average of all these folds. By doing this, we ensure that a hyperparameter search does not overfit to a specific subset of data, and won't be able to generalize to new data.
</p>

<p>
Here, we'll also check whether adding polynomial features (interaction terms) will help our model. Since we're dealing with a linear model on data that seems to not all be non-linear (based on the clusters created previously), it's pretty likely that this will improve our model.
</p>

In [ ]:
pipeline = Pipeline([
    ("cluster", "passhtrough"),
    ("polynomial", "passthrough"),
    ("scale", "passthrough"),
    ("regression", Ridge())
])

params_narrow = {
    "cluster": [t_clusters],
    "polynomial": [PolynomialFeatures(degree=2)],
    "scale": [StandardScaler()],
    "regression__alpha": [0.5, 1.0, 1.5],
    "regression__random_state": [seed, seed + 1]
}

params_wide = {
    "cluster": [
        "passthrough",
        t_clusters
    ],
    "polynomial": [
        "passthrough",
        PolynomialFeatures(degree=2)
    ],
    "scale": [
        MinMaxScaler(),
        StandardScaler()
    ],
    "regression__alpha": [0.5, 1.0, 1.5],
    "regression__random_state": [seed, seed + 1, seed + 2]
}

cv = KFold(n_splits=5, shuffle=True, random_state=seed)
gscv = GridSearchCV(pipeline, param_grid=params_narrow, scoring="r2", cv=cv, n_jobs=1, verbose=1)
gscv = gscv.fit(X_val, y_val)

<p>
Now that we've found the best hyperparameters, let's check the robustness.
We want to check the consistency between scores for each fold, so as to not pick a set of hyperparameters that in reality just did really well on a specific fold and subpar on others.
</p>

<small style="margin-left: 1em">[8]</small>

In [ ]:
## Fetch CV results 

robustness_df = pd.DataFrame(gscv.cv_results_)
robustness_df = robustness_df.sort_values(by=["rank_test_score"]).reset_index()
robustness_df = robustness_df.drop(columns=["mean_fit_time", "std_fit_time", "mean_score_time", "std_score_time", "param_cluster"])

## Rename columns

robustness_df = robustness_df.filter(regex=r"split\d*_test_score").transpose()
robustness_df = robustness_df.rename(columns=lambda v: f"Candidate {v + 1}")
robustness_df = robustness_df.reset_index().drop(columns="index").rename(index=lambda v: f"Split {v + 1}")

## Plot

fig_robustness = make_subplots(rows=2, cols=1, subplot_titles=("Scores for each split", "Correlation between splits"))
fig_scores = px.line(robustness_df, markers=True, labels={ "index": "Split nr.", "value": "R2" })
fig_corr = px.imshow(robustness_df.transpose().corr())

for trace in fig_scores.data: fig_robustness.add_trace(trace, row=1, col=1)
for trace in fig_corr.data: fig_robustness.add_trace(trace, row=2, col=1)

fig_robustness.update_layout(height=800, coloraxis_colorbar=dict(orientation="h", y=-0.2, yanchor="bottom", x=0.5, xanchor="center"), coloraxis=dict(colorscale="Plasma"))
fig_robustness.show()

<p>
We see that the scores for each split are relativily consistent, this is a good sign, because it tells us that we are not heavily relying on a specific set of hyperparameters for a specific fold. In addition, the variance between the candidates is also consistent between splits, which tells us that the change in hyperparameters is consistent across each fold and not overfitting to a specific one.
</p>

<h1>Final Model</h1>

<p>
We see that the model performs the best using clustering, z-score normalization (<code>StandardScaler</code>) and an alpha value of <code>1.5</code> for the Ridge regression.
</p>

<p>
Now that we have found a model, preprocessing steps, and hyperparameters that performs well on our training and validation set, we'll train it once more on the combination of those two sets in a last attempt to increase the performance.
</p>

<p>
Aftwards, we'll create some visualizations showing the final performance.
</p>


In [ ]:
gc.collect()

X_train_val = pd.concat([X_train, X_val], axis=0)
y_train_val = pd.concat([y_train, y_val], axis=0)

model = gscv.best_estimator_
model = model.fit(X_train_val, y_train_val)

model_metrics = dict({
    "Mean Absolute Error": mean_absolute_error(y_test, model.predict(X_test)),
    "Root Mean Squared Error": root_mean_squared_error(y_test, model.predict(X_test)),
    "R2": r2_score(y_test, model.predict(X_test))
})

for k, v in model_metrics.items():
    print(f"{k}: {v:.5f}")

<p>
First off, we see that the R² score is <code>0.98692</code>.
This score tells us how much the variation in our data can be explained by our model.
The best possible score is <code>1.0</code>, so it seems like we're doing pretty good.
</p>

<p>
Let's now also do a scatter plot of the actual vs. predicted values.
</p>

In [ ]:
## Create dataframe with actual/predicted values

df_preds = pd.concat([X_test, y_test], axis=1)
df_preds = df_preds.rename(columns={ "TEMP": "TEMP_ACTUAL" })
df_preds["TEMP_PREDICTED"] = model.predict(X_test)
df_preds["TEMP_DIFF"] = np.absolute(df_preds["TEMP_ACTUAL"] - df_preds["TEMP_PREDICTED"])

## Create figure

fig = px.scatter(df_preds.sample(300, random_state=seed), x="TEMP_ACTUAL", y="TEMP_PREDICTED", color="TEMP_DIFF", labels={
    "TEMP_PREDICTED": "Predicted temperature",
    "TEMP_ACTUAL": "Actual temperature"
})

fig = fig.add_traces(px.line(pd.DataFrame({"x": [0, 30], "y": [0, 30] }), x="x", y="y").data)
fig = fig.update_xaxes(range=[-0.5, 30.5])
fig = fig.update_yaxes(range=[-0.5, 30.5])
fig = fig.update_layout(width=800, height=800, coloraxis_showscale=False)

## Display

fig.show()

<h1>Conclusion</h1>

<p>
We ended up with a model that's quite good for predicting the temperature based on our variables. It helped to include clustering, which is interesting, because it tells us that there's some interactions between our variables that were not "caught" by the Ridge regression, but we needed to "help it".
</p>

<p>
In addition, generating interaction features using polynomials also helped our model quite a bit, which makes sense considering Ridge regression is a linear model and doesn't feature interactions (at least not directly) on its own.
</p>

<p>
To further improve our model, we could try various things, some of which are:
</p>

<ul>
    <li>Detect and tag/remove outliers.</li>
    <li>Find out which interactions affect our model (e.g. by looking at a decision tree).</li>
    <li>Try creating higher degree polynomials.</li>
    <li>Try other models.</li>
</ul>

<h1>Acknowledgements</h1>

<p>
“These data were collected and made freely available by the International Argo Program and the national programs that contribute to it. (<a href="https://argo.ucsd.edu" target="_blank">https://argo.ucsd.edu</a>, <a href="https://www.ocean-ops.org" target="_blank">https://www.ocean-ops.org</a>). The Argo Program is part of the Global Ocean Observing System.”
</p>

<h1>References</h1>

<p>[1]    Argo. <a href="https://argo.ucsd.edu" target="_blank">https://argo.ucsd.edu</a>. 2026.</p>
<p>[2]    Argo Program Office. <i>About Argo</i>. <a href="https://argo.ucsd.edu/about" target="_blank">https://argo.ucsd.edu/about</a> [Online; accessed 26-Februrary-2026]. 2026.</p>
<p>[3]    Wong, Annie P. S., et. al. <i>Argo Data 1999-2019: Two Million Temperature-Salinity
Profiles and Subsurface Velocity Observations From a Global Array of Profiling Floats</i>. <a href="https://www.frontiersin.org/journals/marine-science/articles/10.3389/fmars.2020.00700" target="_blank">https://www.frontiersin.org/journals/marine-science/articles/10.3389/fmars.2020.00700</a> [Online; accessed 26-Februrary-2026]. 2026.</p>
<p>[4]    Argo (2000). <i>Argo float data and metadata from Global Data Assembly Centre (Argo GDAC)</i>. SEANOE. <a href="http://doi.org/10.17882/42182" target="_blank">http://doi.org/10.17882/42182</a></p>
<p>[5]    Wikipedia contributors (2026). <i>El Niño–Southern Oscillation</i>. Wikipedia, The Free Encyclopedia. <a href="https://en.wikipedia.org/w/index.php?title=El_Ni%C3%B1o%E2%80%93Southern_Oscillation&oldid=1341681100" target="_blank">https://en.wikipedia.org/w/index.php?title=El_Ni%C3%B1o%E2%80%93Southern_Oscillation&oldid=1341681100</a> [Online; accessed 5-March-2026</p>
<p>[6]    Wikipedia contributors (2026). <i>Brunt–Väisälä frequency</i>. Wikipedia, The Free Encyclopedia. <a href="https://en.wikipedia.org/w/index.php?title=Brunt%E2%80%93V%C3%A4is%C3%A4l%C3%A4_frequency&oldid=1324127483" target="_blank">https://en.wikipedia.org/w/index.php?title=Brunt%E2%80%93V%C3%A4is%C3%A4l%C3%A4_frequency&oldid=1324127483</a> [Online; accessed 17-March-2026</p>
<p>[7]    Woods Hole Oceanographic Institution (2026). <a href="https://www2.whoi.edu/site/argo/" target="_blank">https://www2.whoi.edu/site/argo/</a> [Online; accessed 18-March-2026</p>
<p>[8]    scikit-learn developers (2025). <i>Statistical comparison of models using grid search</i>. scikit-learn. <a href="https://scikit-learn.org/stable/auto_examples/model_selection/plot_grid_search_stats.html#sphx-glr-auto-examples-model-selection-plot-grid-search-stats-py" target="_blank">https://scikit-learn.org/stable/auto_examples/model_selection/plot_grid_search_stats.html#sphx-glr-auto-examples-model-selection-plot-grid-search-stats-py</a> [Online; accessed 18-March-2026</p>
<p>[9]    Wikipedia contributors (2026). <i>Coefficient of determination</i>. Wikipedia, The Free Encyclopedia. <a href="https://en.wikipedia.org/w/index.php?title=Coefficient_of_determination&oldid=1339942705" target="_blank">https://en.wikipedia.org/w/index.php?title=Coefficient_of_determination&oldid=1339942705</a> [Online; accessed 19-March-2026</p>